# LLM APIs, Reasoning Flows & Prompt-to-Action Pipelines

**Track:** Applied Agent Engineering Foundations  
**Module:** LLM Foundations  
**Environment:** SageMaker Jupyter Notebook  
**Audience:** Software engineers building enterprise AI workflows on AWS

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect securely from SageMaker to Amazon Bedrock.
2. Discover available Bedrock foundation models from code.
3. Use Bedrock LLM APIs for prompt experiments and structured generation.
4. Build a prompt-to-action pipeline with tool/function-calling style behavior.
5. Validate model output before execution.
6. Manage context windows using chunking and budget checks.
7. Add safe error handling and run logging for enterprise usage.

## Instructor flow
- Part A: Secure setup and model discovery
- Part B: Prompt styles and reasoning flows
- Part C: Prompt-to-action pipeline and tool/function calling pattern
- Part D: Context window management
- Part E: Error handling, validation, and mini-hack

## Before you run

This notebook assumes:
- you are running inside **SageMaker Jupyter Notebook**
- your notebook has an **execution role** with Bedrock and S3 read permissions
- Bedrock model access is already enabled for the models you want to test
- your learning files are already uploaded to **S3**
- you are **reading** from S3 only in this notebook; no S3 write-back is required

### Design choice for this lab
This notebook teaches a **model-agnostic prompt-to-action pipeline**.  
That means learners first understand:
- how to call an LLM API
- how to get structured output
- how to validate it
- how to safely call backend functions

Later, the same pattern can be extended to native tool use, agents, workflows, or orchestration frameworks.

In [ ]:
# Uncomment only if your environment is missing any package
# %pip install -q boto3 pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional

import boto3
import pandas as pd
from botocore.exceptions import ClientError


# Get AWS_REGION from the current boto3 session
AWS_REGION = boto3.Session().region_name 

# Bedrock clients
bedrock = boto3.client("bedrock", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Other AWS clients we will use in this notebook
s3 = boto3.client("s3", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)


# Print out the configuration to verify everything is set up correctly
print("AWS Region:", AWS_REGION)

## Step 1 — Show available models in Bedrock

**Goal:** Let learners see what Bedrock exposes in the current region.

**Discuss:**
- model access can vary by account and region
- some models support chat, some embeddings, some image or multimodal inputs
- classroom code should **discover** available models instead of relying on memory

In [ ]:
def list_bedrock_models() -> pd.DataFrame:
    response = bedrock.list_foundation_models()
    rows = []

    for item in response.get("modelSummaries", []):
        rows.append({
            "provider": item.get("providerName"),
            "model_id": item.get("modelId"),
            "model_name": item.get("modelName"),
            "input_modalities": ", ".join(item.get("inputModalities", [])),
            "output_modalities": ", ".join(item.get("outputModalities", [])),
            "streaming": item.get("responseStreamingSupported"),
            "customizations": ", ".join(item.get("customizationsSupported", [])),
            "inference_types": ", ".join(item.get("inferenceTypesSupported", [])),
        })

    df = pd.DataFrame(rows).sort_values(["provider", "model_id"]).reset_index(drop=True)
    return df

try:
    models_df = list_bedrock_models()
    with pd.option_context('display.max_rows', 250):
        display(models_df)
except ClientError as e:
    print("Could not list models. Check Bedrock permissions and regional access.")
    print(e)

## Step 2 — Secure endpoint usage and minimal smoke test

**Goal:** Confirm that SageMaker can call Bedrock safely.

**Secure usage principles:**
- do **not** hardcode keys in notebooks
- use the **SageMaker execution role**
- restrict model IDs through config or an allowlist
- start with a **small, cheap** smoke test
- fail early if the model ID is not approved

**Why this matters:**  
In enterprise settings, the first failure should happen **before** a costly or unsafe request is sent.

In [ ]:
# Choose one classroom model that supports text generation through Converse.
# Available model for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
"""
BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"

print("Default text model:", BEDROCK_TEXT_MODEL)
print("Caller ARN:", sts.get_caller_identity()["Arn"])

In [ ]:
# Call LLM and get a response
response = bedrock_runtime.converse(
    modelId=BEDROCK_TEXT_MODEL,
    system=[{"text": "You are a helpful enterprise AI assistant."}],
    messages=[
        {
        "role": "user",
        "content": [{"text": "In two lines, summarize the plot of the movie Inception."}]}
    ]   ,
    inferenceConfig={
    "maxTokens": 250,
    "temperature": 0.5,
    "topP": 0.9
    }
)

print(response)

In [ ]:
print(response["usage"])

In [ ]:
print(response["ResponseMetadata"])

In [ ]:
print(response["output"])

In [ ]:
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# Now do the same with a helper function that we can reuse for the rest of the notebook
def ask_llm(user_prompt: str,
            system_prompt: str = "You are a helpful enterprise AI assistant.",
            model_id: str = BEDROCK_TEXT_MODEL,
            max_tokens: int = 250,
            temperature: float = 0.2,
            top_p: float = 0.9) -> str:

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature,
            "topP": top_p
        }
    )
    return response["output"]["message"]["content"][0]["text"]


In [ ]:
# Test the helper function with a new prompt
print(ask_llm("In two lines, explain what an LLM API call is."))

In [ ]:
# Change the parameters to see how the response changes
print(ask_llm("In two lines, explain what an LLM API call is.",max_tokens=100, temperature=0.5))

In [ ]:
# Activity: Change prompt parameters
# Task:
# Try the same prompt with:
# 1. temperature = 0
# 2. temperature = 0.7
# 3. max_tokens = 50
# 4. top_p = 0.5
# Compare the outputs.

prompt = "Explain why LLM validation matters in enterprise applications."

In [ ]:
# Activity: 
# Ask the model about a very recent or real-time event.
# Example:
# - "What are the latest news updates today?"
# - "Who won yesterday's IPL match?"
# - "What is happening in global markets right now?"


## Step 3 — Prompt styles and reasoning flows

We will test three common patterns:
1. simple instruction
2. role-based instruction
3. structured reasoning flow

**Explain to learners:**  
Better prompts are not just longer prompts.  
Better prompts make the model's job easier by clarifying:
- role
- task
- output shape
- constraints
- reasoning path

In [ ]:
# Simple instruction prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
print(ask_llm(prompt))

In [ ]:
# Role based prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems."""
print(ask_llm(prompt))

In [ ]:
# Reasoning flow prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems.
            Explain your reasoning step by step."""
print(ask_llm(prompt))

In [ ]:

# Activity: Compare prompt styles
# Task:
# Write one simple prompt and one role-based prompt for the same question.
# Compare which answer is more useful.
# You can also play with temperature and max tokens

# Your code goes here


## System Prompts

In [ ]:
# System message prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = "You are an enterprise AI expert. Please provide a detailed answer."        
print(ask_llm(prompt, system_prompt=system_prompt))

In [ ]:
# Another example of System message prompt style with a more specific system prompt
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are a seasoned enterprise AI consultant with 20 years of experience.
                     You have worked with Fortune 500 companies across various industries to help them implement and validate their AI systems effectively. 
                     Please provide a detailed answer on why validation is crucial in enterprise AI systems, drawing from your extensive experience."""     
print(ask_llm(prompt, system_prompt=system_prompt))

In [ ]:
# More complex prompt styles with few-shot examples can also be used
prompt = "You are an enterprise AI expert."
system_prompt = """You are an enterprise AI expert.
Here are some examples of how to answer questions about enterprise AI systems:                  
Q: Why is validation important in enterprise AI systems?
A: Validation is crucial in enterprise AI systems to ensure that the models are performing as expected,
    to identify any biases or issues before deployment, and to maintain trust with stakeholders by demonstrating that the AI system is reliable and effective.  
Q: What are some common techniques for validating enterprise AI systems?    
A: Common techniques for validating enterprise AI systems include cross-validation, A/B testing, monitoring performance metrics in production, and conducting regular audits to check for bias and drift."""
print(ask_llm(prompt, system_prompt=system_prompt))


In [ ]:
# Change system prompt to agent tell the model it is an agent that needs to answer the question and take action if needed.
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
print(ask_llm(prompt, system_prompt=system_prompt))

In [ ]:
#  Activity: Rewrite system prompt
# Task:
# Change the system prompt so the model answers as:
# 1. QA lead
# 2. Product manager
# 3. Security reviewer

user_prompt = "What risks should we check before deploying an LLM application?"

# Your code goes here 

# LLM Interaction Logging & Observability Framework

Build a reusable LLM interface that not only generates responses but also captures structured telemetry for monitoring, cost tracking, and analysis.

In [ ]:
import os
import pandas as pd
from datetime import datetime

LOG_CSV_PATH = "llm_prompt_log.csv"

def ask_llm(
    user_prompt: str,
    asked_by: str,
    system_prompt: str = "You are a helpful enterprise AI assistant.",
    model_id: str = BEDROCK_TEXT_MODEL,
    max_tokens: int = 250,
    temperature: float = 0.2,
    csv_path: str = LOG_CSV_PATH
) -> str:
    """
    Call the Bedrock model, return the answer,
    and log prompt/response details into a CSV using pandas.
    """

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )

    # Extract answer text
    answer = response["output"]["message"]["content"][0]["text"]

    # Extract usage safely
    usage = response.get("usage", {})
    input_tokens = usage.get("inputTokens", None)
    output_tokens = usage.get("outputTokens", None)
    total_tokens = usage.get("totalTokens", None)

    # Create one log row
    new_row = pd.DataFrame([{
        "timestamp": timestamp,
        "asked_by": asked_by,
        "model_id": model_id,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response_text": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "max_tokens": max_tokens,
        "temperature": temperature
    }])

    # Append to existing CSV or create new one
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        updated_df = pd.concat([existing_df, new_row], ignore_index=True)
    else:
        updated_df = new_row

    updated_df.to_csv(csv_path, index=False)

    return answer

In [ ]:
user_prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
user = "user@123"
ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

In [ ]:
# Activity:
# Please try asking the model a few more questions with different system prompts and parameters to generate more rows in the log for analysis.
# Inspect the LLM log
# Task:
# Display the CSV log and compare token usage across prompts.

# user_prompt = ""
# system_prompt = ""
# user = ""
# ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

log_df = pd.read_csv(LOG_CSV_PATH)
display(log_df)

## Reflection checkpoint
Discuss pros and cons of prompt style before move to next section

## Step 4 — Build a prompt-to-action pipeline

A production-safe workflow often looks like this:
1. understand the request
2. convert it into a structured plan
3. validate the plan
4. execute only the allowed action
5. call a controlled backend function
6. log the result

This is the core **tool/function calling pattern** we want learners to understand.

### Important note
This notebook uses a **model-agnostic structured JSON plan**.  
That keeps the concept simple and portable across models.

In [ ]:
ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_backend_action": true or false,
  "action_name": "<action or none>",
  "arguments": { ... },
  "reason_for_action": "<one sentence>"
}

Allowed action names:
- create_ticket
- lookup_user_record
- none

Examples:
User: "Create an IT ticket for VPN access issue"
Output:
{"intent":"create_it_ticket","needs_backend_action":true,"action_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_action":"A ticket must be created in the backend system."}

User: "Find the team and location for ananya"
Output:
{"intent":"lookup_user_record","needs_backend_action":true,"action_name":"lookup_user_record","arguments":{"user_name":"ananya"},"reason_for_action":"The answer should come from the CSV file loaded from S3."}
'''


In [ ]:
# Function to extract JSON object from LLM response
def extract_json_object(text: str) -> dict:
    return json.loads(text.strip())

# Function to plan action based on user request
def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_prompt=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_prompt="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0,
        asked_by = "user@123"
    )
    return extract_json_object(raw)

In [ ]:
# Test the action planner with a sample user request
plan_action("Create an IT ticket for laptop replacement")

In [ ]:
# Test the action planner with another sample user request
plan_action("Find the team and location for ananya")

In [ ]:
import pandas as pd
# Read data from S3 to be used for backend action execution
S3_DATA_BUCKET = "rag-demo-docs-sri"
S3_FILE_PATH = "training_user_directory.csv"

def read_csv_from_s3(bucket: str, file_path: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=file_path)
    return pd.read_csv(obj["Body"])

df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)
print(df.head())

In [ ]:
# Function to execute the planned action
def execute_plan(plan: dict) -> dict:
    action_name = plan["action_name"]

    if action_name == "lookup_user_record":
        user_name = plan["arguments"]["user_name"]
        df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)

        match = df[df["user_name"].str.lower() == user_name.lower()]

        if match.empty:
            return {
                "status": "not_found",
                "message": f"No user found for '{user_name}'."
            }

        row = match.iloc[0]

        return {
            "status": "success",
            "message": "User found.",
            "user_record": {
                "user_name": row["user_name"],
                "full_name": row["full_name"],
                "team": row["team"],
                "location": row["location"],
                "manager": row["manager"],
                "laptop_type": row["laptop_type"]
            }
        }

    return {
        "status": "no_action",
        "message": "No backend action required."
    }


In [ ]:
# run the full pipeline with a sample user request
def run_prompt_to_action_pipeline(user_request: str) -> dict:
    plan = plan_action(user_request)
    result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "result": result
    }

In [ ]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for ananya")
print(json.dumps(response, indent=2))

In [ ]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for karthik")
print(json.dumps(response, indent=2))